# Lab 16: Data Science Agent avec GCP BigQuery

**Navigation** : [Lab 15 <<](../Day6-MLE-Star/Lab15-Kaggle-Challenge.ipynb) | [Index](../../README.md) | [>> Lab 17](Lab17-Final-Project.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Comprendre l'architecture de l'agent Data Science Google
2. Explorer NL2SQL et NL2Py pour BigQuery
3. Intégrer BQML pour le machine learning dans le data warehouse
4. Architecturer un agent de production sur le cloud

### Prérequis
- Lab 15 (MLE-STAR) complété
- Compte GCP (optionnel pour ce lab théorique)
- Connaissance de SQL

### Durée estimée : 30-40 minutes

## 1. Configuration

In [1]:
import sys
sys.path.insert(0, '..')

import json
import re
from typing import List, Dict, Optional
from dataclasses import dataclass

from config import get_settings
from utils import LLMClient

ModuleNotFoundError: No module named 'config'

Chargement des parametres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

NameError: name 'get_settings' is not defined

## 2. NL2SQL Translator

In [3]:
@dataclass
class SQLQuery:
    sql: str
    explanation: str
    tables_used: List[str]

class NL2SQLTranslator:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def translate(self, question: str, schema: Dict) -> SQLQuery:
        schema_desc = self._format_schema(schema)
        prompt = f"""Convertis cette question en SQL BigQuery.

SCHEMA:
{schema_desc}

QUESTION: {question}

EXPLANATION: [explication]
SQL:
```sql
[requete]
```"""
        response = self.llm.generate(prompt, temperature=0.2)
        explanation = ''
        sql = ''
        if 'EXPLANATION:' in response:
            match = re.search(r'EXPLANATION:\s*(.+?)(?=SQL:|$)', response, re.DOTALL)
            explanation = match.group(1).strip() if match else ''
        sql_match = re.search(r'```sql\s*(.*?)\s*```', response, re.DOTALL)
        sql = sql_match.group(1).strip() if sql_match else ''
        return SQLQuery(sql=sql, explanation=explanation, tables_used=[])

    def _format_schema(self, schema: Dict) -> str:
        return '\n'.join([f'Table {t}: {cols}' for t, cols in schema.items()])

NameError: name 'LLMClient' is not defined

## 3. NL2Py Translator

In [4]:
@dataclass
class PythonCode:
    code: str
    explanation: str

class NL2PyTranslator:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def translate(self, question: str, data_context: str) -> PythonCode:
        prompt = f"""Genere du code Python pour: {question}

CONTEXTE: {data_context}

EXPLANATION: [explication]
CODE:
```python
[code]
```"""
        response = self.llm.generate(prompt, temperature=0.2)
        explanation = ''
        code = ''
        if 'EXPLANATION:' in response:
            match = re.search(r'EXPLANATION:\s*(.+?)(?=CODE:|$)', response, re.DOTALL)
            explanation = match.group(1).strip() if match else ''
        code_match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
        code = code_match.group(1).strip() if code_match else ''
        return PythonCode(code=code, explanation=explanation)

NameError: name 'LLMClient' is not defined

## 4. Data Science Agent

In [5]:
class DataScienceAgent:
    def __init__(self):
        self.llm = LLMClient()
        self.nl2sql = NL2SQLTranslator(self.llm)
        self.nl2py = NL2PyTranslator(self.llm)

    def analyze(self, question: str, schema: Dict, mode: str = 'sql') -> Dict:
        print(f'[AGENT] Question: {question}')
        print(f'[AGENT] Mode: {mode}')
        if mode == 'sql':
            result = self.nl2sql.translate(question, schema)
            return {'type': 'sql', 'query': result.sql, 'explanation': result.explanation}
        elif mode == 'python':
            data_context = self.nl2sql._format_schema(schema)
            result = self.nl2py.translate(question, data_context)
            return {'type': 'python', 'code': result.code, 'explanation': result.explanation}
        return {'error': 'Mode non supporte'}

## 5. Test avec Schema Simule

In [6]:
schema = {
    'sales': ['date', 'product', 'region', 'quantity', 'revenue'],
    'customers': ['customer_id', 'name', 'segment', 'signup_date'],
    'products': ['product_id', 'name', 'category', 'price']
}

print('Schema BigQuery:')
for table, cols in schema.items():
    print(f'  {table}: {cols}')

Schema BigQuery:
  sales: ['date', 'product', 'region', 'quantity', 'revenue']
  customers: ['customer_id', 'name', 'segment', 'signup_date']
  products: ['product_id', 'name', 'category', 'price']


Test NL2SQL : traduction de requetes naturelles en SQL.

In [7]:
# Test NL2SQL
agent = DataScienceAgent()
question = 'Quel est le revenu total par region?'
result = agent.analyze(question, schema, mode='sql')

print('\\n' + '='*50)
print('RESULTAT NL2SQL:')
print('='*50)
print(f'Explication: {result.get("explanation", "N/A")}')
print(f'SQL: {result.get("query", "N/A")}')

NameError: name 'LLMClient' is not defined

Test NL2Py : generation de code Python a partir de langage naturel.

In [8]:
# Test NL2Py
question2 = 'Calcule la moyenne des revenus par mois'
result2 = agent.analyze(question2, schema, mode='python')

print('\\n' + '='*50)
print('RESULTAT NL2Py:')
print('='*50)
print(f'Explication: {result2.get("explanation", "N/A")}')
print(f'Code: {result2.get("code", "N/A")[:300]}...')

NameError: name 'agent' is not defined

## 6. Resume du Lab
### Architecture Google Data Science Agent
1. NL2SQL: Question naturelle vers BigQuery
2. NL2Py: Question naturelle vers Python
3. BQML: Modeles ML dans BigQuery
### Prochaine etape
Lab 17: Projet Final

## Exercice : Agent Data Science Personnalise

Concevez un agent capable de repondre a des questions sur votre propre schema de donnees.

### Objectifs
1. Definir un schema de donnees realiste
2. Tester NL2SQL et NL2Py avec des questions complexes
3. Analyser les limites de la traduction automatique
4. Proposer des ameliorations

### Instructions



In [9]:
# TODO: Definissez votre schema (ex: e-commerce, sante, finance)
mon_schema = {
    'commandes': ['id', 'client_id', 'date', 'montant', 'statut'],
    'clients': ['id', 'nom', 'email', 'date_inscription', 'segment'],
    'produits': ['id', 'nom', 'categorie', 'prix', 'stock'],
    'lignes_commande': ['id', 'commande_id', 'produit_id', 'quantite', 'prix_unitaire']
}

# TODO: Creez 5 questions de complexite croissante
mes_questions = [
    "Quelle est la commande la plus elevee?",  # Simple
    "...",  # Jointure
    "...",  # Agregation temporelle
    "...",  # Sous-requete
    "..."   # Analyse complexe
]

# TODO: Testez chaque question avec les deux modes
agent = DataScienceAgent()
for q in mes_questions:
    print(f"\\nQUESTION: {q}")
    sql_result = agent.analyze(q, mon_schema, mode='sql')
    py_result = agent.analyze(q, mon_schema, mode='python')
    
    print(f"SQL: {sql_result.get('query', 'N/A')[:100]}...")
    print(f"Python: {py_result.get('code', 'N/A')[:100]}...")

# TODO: Analysez les echecs et proposez des corrections

NameError: name 'LLMClient' is not defined

### Questions d'analyse
- Quels types de questions echouent le plus souvent ?
- Le mode SQL ou Python est-il plus robuste ?
- Comment ameliorer le contexte fourni au LLM ?
